In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import sys
sys.path.append('..')
sys.path.append('../..')
import pandas as pd
from model import VAE, plus_encode_data
import seaborn as sns
from sklearn.mixture import GaussianMixture
import joblib
import input_pipeline
from sklearn.manifold import TSNE

In [ ]:
def load_model_at_T(t, folder,  latent_dim = 200):
    model = VAE(latent_dim)
    dummy_input = tf.zeros((1, 32, 32, 1))
    model(dummy_input)
    model.load_weights(f'{folder}/{t:.1f}/vae{t:.1f}.h5')
    return model

folder = ""

# Gaussian mixture fitting

In [ ]:

temps = np.arange(2.0, 3.05, 0.1)
models = {t: load_model_at_T(t, folder) for t in temps}

In [ ]:
gm_vars = GaussianMixture(n_components=2, random_state=0)
gm_means = GaussianMixture(n_components=1, random_state=0)

for temp in temps:
    model = models[temp]
    batch_size = 100
    
    trainset_path = f"../../GetData/Python/Data/Data{temp:.1f}.tfrecord"
    train_set = input_pipeline.dataset_tfrecord_pipeline(trainset_path, flatten=False, batch_size=batch_size)

    # data mc trainset
    data_var = []
    data_mean = []
    for batch in train_set:
        data_mc = batch
        data_plus = plus_encode_data(data_mc)
        mean, var = model.encode(data_plus)
        data_var.append(var)
        data_mean.append(mean)
    
    vars = np.concatenate(data_var, axis=0)
    means = np.concatenate(data_mean, axis=0)

    gm_vars.fit(vars)
    gm_means.fit(means)

    # Save the GMM models
    joblib.dump(gm_vars, f"{folder}/{temp:.1f}/gm_vars.pkl")
    joblib.dump(gm_means, f"{folder}/{temp:.1f}/gm_means.pkl")
    
    print(f"Temperature: {temp:.1f}")

# Tsne

In [ ]:
temp = 2.3
model = load_model_at_T(temp, folder)
batch_size = 100

trainset_path = f"../../GetData/Python/Data/Data{temp:.1f}.tfrecord"
train_set = input_pipeline.dataset_tfrecord_pipeline(trainset_path, flatten=False, batch_size=batch_size)

# data mc trainset
data_var = []
data_mean = []
for batch in train_set:
    data_mc = batch
    data_plus = plus_encode_data(data_mc)
    mean, var = model.encode(data_plus)
    data_var.append(var)
    data_mean.append(mean)

vars = np.concatenate(data_var, axis=0)
means = np.concatenate(data_mean, axis=0)

In [ ]:
tsne_mean = TSNE(n_components=2, perplexity=50, n_iter=10000, verbose=1, random_state=123)
means_sampled = means[np.random.choice(means.shape[0], 5000, replace=False)]
v = tsne_mean.fit_transform(means_sampled) 

df_mean = pd.DataFrame()
df_mean["x"] = v[:,0]
df_mean["y"] = v[:,1]

In [ ]:
sns.set_theme(rc={
    'font.size': 15,
    'axes.titlesize': 15,
    'axes.labelsize': 15,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 15,
    'figure.figsize': [6, 4.5],
})

In [ ]:
sns.kdeplot(data=df_mean, x='x', y='y', cmap='viridis', fill=True, cbar=True)
plt.xlabel("t-SNE x")
plt.ylabel("t-SNE y")
plt.savefig(f"{folder}/plots/means_tsne.svg")

In [ ]:
tsne_var = TSNE(n_components=2, perplexity=50, n_iter=10000, verbose=1, random_state=123)
vars_sampled = vars[np.random.choice(vars.shape[0], 5000, replace=False)]
v = tsne_var.fit_transform(vars_sampled) 
df_var = pd.DataFrame()
df_var["x"] = v[:,0]
df_var["y"] = v[:,1]

In [ ]:
sns.kdeplot(data=df_var, x='x', y='y', cmap='viridis', fill=True, cbar=True)
plt.xlabel("t-SNE x")
plt.ylabel("t-SNE y")
plt.savefig(f"{folder}/plots/vars_tsne.svg")

In [ ]:
gm_vars = GaussianMixture(n_components=2, random_state=0)
gm_vars.fit(vars_sampled)
labels = gm_vars.predict(vars_sampled)
df_var['label'] = labels
sns.kdeplot(data=df_var, x='x', y='y', cmap='viridis', fill=True, cbar=False, hue='label', alpha=0.5)
plt.xlabel("t-SNE x")
plt.ylabel("t-SNE y")
plt.savefig(f"{folder}/plots/vars_tsne_gm.svg")